[Homework 2]('https://colab.research.google.com/github/Jaguar838/llm-zoomcamp/blob/main/HW/2026/02-vector-search/hw-02.ipynb')

In [ ]:
# Install necessary Python packages for data processing, search, LLM interaction, environment variable management, and Git data reading.
%pip install requests minsearch groq gitsource sqlitesearch huggingface-hub tqdm onnxruntime tokenizers

## Homework: Vector Search

In this homework, we put what we learned in Module 2 into practice.

We'll first turn text into vectors, then search by similarity.
We'll also learn something new and see how to combine vector search with keyword search. We'll skip the RAG part and focus solely on search.

Like in homework 1, our knowledge base is the course lessons themselves.
Each module has a `lessons/` folder of numbered markdown
pages, and we pull them from GitHub. We use the same commit, `8c1834d`,
so everyone works with the exact same 72 pages.

> It's possible your answers won't match exactly. If so, select the closest one.

## Setup

In this homework we won't use the same approach for embedding as in the
module. That is, we won't use the sentence-transformers library. Instead,
we'll use the lightweight embedding approach with the ONNX `Embedder`.

Both approaches produce identical vectors, but the ONNX runtime is far
lighter. It needs no PyTorch and no CUDA, which makes the install about
30x smaller and lets it run anywhere, including a basic Codespace. We
skimmed through it in the lesson and said we'd cover it in the homework -
so here we are.

We prepare the environment the same way as in the module's
[ONNX Runtime](../../../02-vector-search/lessons/09-onnx-embedder.md)
lesson.

Create a fresh project and install the dependencies:

```bash
mkdir llm-zoomcamp-hw2 && cd llm-zoomcamp-hw2
uv init --no-workspace
uv add onnxruntime tokenizers numpy tqdm minsearch gitsource
uv add --dev huggingface-hub jupyter
```

We also need two helper scripts from the `embed/` directory of the course
repo:

- [`download.py`](https://github.com/DataTalksClub/llm-zoomcamp/blob/main/02-vector-search/embed/download.py)
(fetches an ONNX model from HuggingFace) and
- [`embedder.py`](https://github.com/DataTalksClub/llm-zoomcamp/blob/main/02-vector-search/embed/embedder.py) (the `Embedder` class with an `encode` interface)

Let's download them:

```bash
PREFIX=https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/02-vector-search/embed
wget $PREFIX/download.py
wget $PREFIX/embedder.py
```

By default `download.py` fetches `Xenova/all-MiniLM-L6-v2`, the ONNX
version of the `all-MiniLM-L6-v2` model from the lessons:

```bash
uv run python download.py
```

Now we're ready to do the homework.

In [3]:
%%bash
PREFIX=https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/02-vector-search/embed
wget $PREFIX/download.py
wget $PREFIX/embedder.py
python download.py

  saved models/Xenova/all-MiniLM-L6-v2/tokenizer.json
  saved models/Xenova/all-MiniLM-L6-v2/model.onnx


--2026-06-29 04:02:36--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/02-vector-search/embed/download.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.108.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1376 (1.3K) [text/plain]
Saving to: ‘download.py’

     0K .                                                     100% 63.1M=0s

2026-06-29 04:02:36 (63.1 MB/s) - ‘download.py’ saved [1376/1376]

--2026-06-29 04:02:36--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/02-vector-search/embed/embedder.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.108.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
L

In [24]:
!ls

download.py  embedder.py  models  __pycache__  sample_data


## Q1. Embedding a query

Embed the following query:

> How does approximate nearest neighbor search work?

The embedder returns a vector of 384 numbers. What's the first value
(`v[0]`)?

* -0.31
* -0.02
* 0.12
* 0.44

In [9]:
# It looks like the embedder module isn't being found. I'll add the current directory to the Python path to ensure it can be imported correctly.

from embedder import Embedder

model_path = 'models/Xenova/all-MiniLM-L6-v2'
embedder = Embedder(model_path)

query = "How does approximate nearest neighbor search work?"
v = embedder.encode(query)

print(f"Vector length: {len(v)}")
print(f"Q1: v[0] = {v[0]:.2f}")

Vector length: 384
Q1: v[0] = -0.02


## Loading the data

We pull the lesson pages from the course repository, the same way as in
homework 1. We pin to commit `8c1834d` so everyone works with the same
data.

```python
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]
```

Each document is a dictionary with a `filename` and `content`, and there
are 72 pages.

In [4]:
from gitsource import GithubRepositoryDataReader

# Initialize GithubRepositoryDataReader to read markdown files from a specific GitHub repository commit.
# It filters for files in the 'lessons/' path.
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

# Read the files matching the criteria.
files = reader.read()

In [5]:
documents = [file.parse() for file in reader.read()]
len(documents)

72

## Q2. Cosine similarity

The embedder returns normalized vectors, so the dot product between two
of them is their cosine similarity.

Take the page `02-vector-search/lessons/07-sqlitesearch-vector.md`, embed
its `content`, and compute the cosine similarity with the query vector
from Q1. What do you get?

* 0.07
* 0.37
* 0.68
* 0.92


In [6]:

# 1. Get Query Vector (v) from Q1
from embedder import Embedder

model_path = 'models/Xenova/all-MiniLM-L6-v2'
embedder = Embedder(model_path)

query = "How does approximate nearest neighbor search work?"
v = embedder.encode(query)

# 2. Fetch the specific document content
target_filename = '02-vector-search/lessons/07-sqlitesearch-vector.md'

doc = next((d for d in documents if d['filename'] == target_filename), None)

if doc is None:
    print(f"Document {target_filename} not found!")

# 3. Embed the document content
v_doc = embedder.encode(doc['content'])

# 4. Compute cosine similarity (dot product since vectors are normalized)
similarity = v.dot(v_doc)

print(f"Target file: {doc['filename']}")
print(f"Q2: similarity: {similarity:.2f}")

Target file: 02-vector-search/lessons/07-sqlitesearch-vector.md
Q2: similarity: 0.36


## Q3. Chunking and search by hand

A full page covers several topics, which waters down its embedding.

We chunk the pages the same way as in homework 1:

```python
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)
```

We embed every chunk's `content` with `encode_batch`, stack the vectors
into a matrix `X`, and score the Q1 query against all chunks:

```python
scores = X.dot(v)
```

Which file does the highest-scoring chunk belong to (its `filename`)?

* `02-vector-search/lessons/03-embeddings-dataset.md`
* `02-vector-search/lessons/06-rag-vector.md`
* `02-vector-search/lessons/07-sqlitesearch-vector.md`
* `02-vector-search/lessons/09-onnx-embedder.md`

In [10]:
from gitsource import chunk_documents

# Chunk the documents into smaller pieces with a specified size and step for overlapping.
chunks = chunk_documents(documents, size=2000, step=1000)

In [22]:
chunks[2]

{'start': 2000,
 'content': 'wrong.\n\n## The project\n\nRAG solves these problems by giving the LLM relevant documents at\nquestion time. We don\'t hope the model memorized the answer. We\nretrieve the right information and hand it to the LLM, and the model\ngenerates a grounded response. This lets us inject knowledge the model\nnever saw during training. That\'s why RAG is still the most common way\npeople use LLMs in the industry.\n\nTo make this concrete, we build a FAQ agent for our course. A student\nasks something like "when does the course start?" and the agent answers\nfrom the FAQ data we prepared.\n\nThis module has two parts.\n\nIn Part 1 (the next 9 lessons) we will:\n\n- Understand what RAG is and how it works\n- Build a search engine over a real FAQ dataset\n- Write a prompt that combines the user\'s question with search results\n- Wire it all together into a working RAG pipeline\n- Split ingestion and query into separate processes\n\nIn Part 2, we make the pipeline agen

In [14]:
# Print the total number of chunks created.
print(f"Q4: {len(chunks)}")

Q4: 295


In [12]:
query = "How does approximate nearest neighbor search work?"
v = embedder.encode(query)

In [17]:
import numpy as np

# Extract content from all chunks into a list
chunk_contents = [chunk['content'] for chunk in chunks]

# Embed all chunks using encode_batch
X = embedder.encode_batch(chunk_contents)

scores = X.dot(v)

In [18]:
X

array([[-0.08756474,  0.0183638 , -0.08122425, ...,  0.0305382 ,
        -0.02172767,  0.03277498],
       [ 0.02436192, -0.10619479,  0.03307313, ...,  0.01430086,
        -0.0012554 ,  0.04325694],
       [-0.01780481,  0.03103092,  0.00856104, ...,  0.02220219,
        -0.03375532,  0.04288225],
       ...,
       [ 0.00980345,  0.04912257,  0.01207493, ..., -0.09453998,
        -0.06321277,  0.04775796],
       [-0.03622024,  0.06821856, -0.01540895, ..., -0.00271633,
         0.01875558,  0.01007469],
       [-0.02975661, -0.00552575, -0.03531848, ...,  0.01044234,
         0.02297965, -0.01966068]])

In [19]:
scores

array([ 3.15187611e-01,  2.01479664e-01,  5.90559037e-02, -7.67661288e-02,
        1.18452466e-01, -1.41697012e-01, -2.81406495e-02, -4.65669117e-02,
       -2.06994543e-02, -6.07744147e-02,  2.13273769e-01,  8.87600958e-02,
       -1.97269268e-02,  3.11629985e-01,  5.51034635e-01,  2.04008152e-01,
        2.12515802e-01,  1.93649107e-01,  2.51961267e-01,  1.31078610e-01,
        2.59120607e-01,  7.63816369e-02,  9.59193203e-02,  9.81471228e-03,
       -3.59107168e-02,  1.01211406e-02,  1.10786940e-01, -9.90259915e-02,
       -3.71170645e-02,  7.59057333e-02, -3.35340234e-02,  8.86841484e-03,
        1.02636448e-01,  6.89615413e-02,  1.29408854e-01,  2.57709121e-01,
        3.23680576e-01,  1.06350076e-01,  5.61891208e-02,  2.34017441e-01,
        1.97954358e-01,  9.64296342e-02,  1.93709934e-01,  2.16719269e-01,
        3.48340450e-01,  5.10906541e-02,  2.05212833e-01,  1.05416182e-01,
       -3.25432660e-02,  4.94665347e-02,  2.38574873e-01, -3.44207304e-02,
        1.82165430e-01,  

In [23]:
max_score_index = np.argmax(scores)
highest_scoring_chunk = chunks[max_score_index]

print(f"Q3: {highest_scoring_chunk['filename']}")

Q3: 02-vector-search/lessons/07-sqlitesearch-vector.md


## Q4. Vector search with minsearch

We've done vector search by hand, which is good for learning, but it's not
what we do in practice. In practice we use libraries.

Let's use `VectorSearch` from minsearch and run a search for the following
query:

> What metric do we use to evaluate a search engine?

Which file is the `filename` of the first result?

* `02-vector-search/lessons/04-vector-search.md`
* `04-evaluation/lessons/05-search-metrics.md`
* `04-evaluation/lessons/13-llm-as-judge.md`
* `05-monitoring/lessons/04-metrics.md`

In [26]:
from minsearch import VectorSearch

In [ ]:
import numpy as np

# Initialize custom VectorSearch with keyword_fields
index_vector = VectorSearch(keyword_fields=["filename"])

# Fit the vector search index with the pre-computed embeddings (X) and chunks (payload)
index_vector.fit(vectors=X, payload=chunks)

# Define the query for Q4
q4_query = "What metric do we use to evaluate a search engine?"

# Embed the query using the existing embedder
q4_query_vector = embedder.encode(q4_query)

# Perform the vector search
vector_results_q4 = index_vector.search(
    query_vector=q4_query_vector,
    num_results=1
)

# Get the filename of the first result
q4_filename = vector_results_q4[0]['filename']

print(f"Q4: {q4_filename}")

In [ ]:
index_sql.count()

72

In [ ]:
import os

file_path = 'lesson_llm_course.db'
file_size_mb = os.path.getsize(file_path) / 1024**2

print(f"File size of {file_path}: {file_size_mb:.2f} MB)")

File size of lesson_llm_course.db: 1.02 MB)


In [ ]:
results = index_sql.search(query, num_results=72)
sql_result_search = [doc["filename"] for doc in results]

In [ ]:
print(f"Count files: {len(sql_result_search)}")
print(f"Q2: {sql_result_search[0]}")

Count files: 59
Q2: 01-agentic-rag/lessons/14-agentic-loop.md


In [ ]:
import os
from groq import Groq
from google.colab import userdata

api_key_groq = userdata.get('GROQ_API_KEY')

# Initialize the Groq client with the retrieved API key.
client_groq = Groq(api_key=api_key_groq)

In [ ]:
# Send a chat completion request to the Groq API using the 'llama-3.1-8b-instant' model.
chat_completion = client_groq.chat.completions.create(
    messages=[
        {
            "role": "user",
            "content": "Explain the importance of fast language models",
        }
    ],
    model="llama-3.1-8b-instant",
)

# Print the content of the assistant's response.
print(chat_completion.choices[0].message.content)

Fast language models have gained significant attention in the field of natural language processing (NLP) in recent years due to their ability to efficiently process and generate large volumes of text data. The importance of fast language models can be seen in several areas:

1. **Efficient Inference**: Fast language models can perform inference tasks much faster than traditional language models, which is crucial when dealing with large-scale applications such as chatbots, voice assistants, and text summarization systems. This efficiency enables faster response times, improved user experience, and increased scalability.
2. **Real-Time Processing**: Fast language models can process text data in real-time, allowing for applications such as live chat, sentiment analysis, and text classification to operate seamlessly. This is particularly important for applications that require timely responses to user input.
3. **Large-Scale Deployments**: Fast language models can be deployed at scale, ena

In [ ]:
# Define the system instructions for the RAG assistant.
INSTRUCTIONS = '''
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
'''

# Define the prompt template that structures the question and context for the LLM.
PROMPT_TEMPLATE = '''
QUESTION: {question}

CONTEXT:
{context}
'''.strip()

In [ ]:
!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py

--2026-06-21 05:56:18--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.109.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2134 (2.1K) [text/plain]
Saving to: ‘rag_helper.py’

rag_helper.py       100%[===================>]   2.08K  --.-KB/s    in 0s      

2026-06-21 05:56:18 (27.9 MB/s) - ‘rag_helper.py’ saved [2134/2134]



In [ ]:
from dataclasses import dataclass
from rag_helper import RAGBase as OriginalRAGBase

# Define a dataclass to store the result of a RAG query, including the answer and token counts.
@dataclass
class RAGResult:
    answer: str
    input_tokens: int
    output_tokens: int

# Define the RAGBase class by inheriting from OriginalRAGBase and adapting it for our schema.
class RAGBase(OriginalRAGBase):

    # self,
    # index,
    # llm_client,
    # instructions=INSTRUCTIONS,
    # prompt_template=PROMPT_TEMPLATE,

    # Method to perform a search using the provided index.
    # Adapted for the filename/content schema.
    def search(self, query, num_results=5):
        return self.index.search(query, num_results=num_results)

    # Method to build context string from search results.
    # Adapted for the filename/content schema.
    def build_context(self, search_results):
        lines = []
        for doc in search_results:
            lines.append(f"Filename: {doc['filename']}")
            lines.append(doc['content'])
            lines.append('')
        return '\n'.join(lines).strip()

    # Method to interact with the LLM.
    # Overridden to use chat.completions.create and return the full response object for token counting.
    def llm(self, prompt):
        input_messages = [
            {"role": "system", "content": self.instructions},
            {"role": "user", "content": prompt}
        ]

        response = self.llm_client.chat.completions.create(
            model=self.model,
            messages=input_messages
        )

        return response

    # Main RAG method to perform the entire RAG process.
    # Overridden to return a RAGResult with token counts.
    def rag(self, query):
        search_results = self.search(query)
        # print(search_results)
        prompt = self.build_prompt(query, search_results)
        response = self.llm(prompt)


        # Extract the answer and token usage from the LLM response.
        answer = response.choices[0].message.content
        input_tokens = response.usage.prompt_tokens
        output_tokens = response.usage.completion_tokens

        return RAGResult(answer=answer, input_tokens=input_tokens, output_tokens=output_tokens)

In [ ]:
# Initialize RAGBase with the document index, Groq client, and specified model.
rag = RAGBase(
    index=index_sql,
    llm_client=client_groq,
    model="llama-3.3-70b-versatile"
)
# Run the RAG query.
result = rag.rag(query)
# Print the answer from the RAG result.
print(result.answer)
# Store the input tokens for Q3.
Q3_input_tokens = result.input_tokens
# Print the input tokens used.
print("Input tokens:", Q3_input_tokens)

The provided context includes multiple lessons and code snippets related to building an agentic RAG (Retrieval-Augmented Generator) pipeline. To answer your question about how the agentic loop keeps calling the model until it stops, we can look at the relevant code snippet:

```python
while True:
    print(f"iteration #{it}...")
    has_function_calls = False

    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=messages,
        tools=[search_tool],
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == "function_call":
            print("function_call:", item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == "message":
            print("ASSISTANT:")
            print(item.content[0].text)

    it = it + 1
    if has_function_calls == False:
        break
```

In this code, the agen

In [ ]:
print(rag.build_prompt)

<bound method RAGBase.build_prompt of <__main__.RAGBase object at 0x7bcce41875f0>>


In [ ]:
# Print the input and output tokens for Q3.
print(f"""Q3:
input tokens: {Q3_input_tokens}
output tokens: {result.output_tokens}""")

Q3:
input tokens: 7044
output tokens: 285


## Q5. Text search vs vector search

Vector search matches by meaning, keyword search by exact words.

Let's compare them. Index
the same chunks with `Index` from minsearch. Use `content` as a
text field.

Run both searches for this query:

> How do I store vectors in PostgreSQL?

Take the top 5 results from each method. Which file shows up in the
vector results but not in the text results?

* `02-vector-search/lessons/01-intro.md`
* `02-vector-search/lessons/02-embeddings.md`
* `02-vector-search/lessons/08-pgvector.md`
* `03-orchestration/lessons/05-rag.md`


In [ ]:
import time
from sqlitesearch import TextSearchIndex

# Initialize a new TextSearchIndex for chunks
chunk_index_sql = TextSearchIndex(
    text_fields=["content"],
    keyword_fields=["filename"],
    db_path="lesson_llm_course_chunks.db" # Using a new database file for chunks
)

# Add each chunk to the new SQL index
for i, chunk in enumerate(chunks):
    chunk_index_sql.add(chunk)
    # print(f"Added chunk {i+1} from {chunk['filename']}...")
    # time.sleep(0.01) # Small delay to avoid overwhelming the system if many chunks

chunk_index_sql.close()
print("Done. Chunk index saved to lesson_llm_course_chunks.db")

Done. Chunk index saved to lesson_llm_course_chunks.db


In [ ]:
import os

file_path_chunks = 'lesson_llm_course_chunks.db'
file_size_bytes_chunks = os.path.getsize(file_path_chunks)
file_size_kb_chunks = file_size_bytes_chunks / 1024
file_size_mb_chunks = file_size_kb_chunks / 1024

print(f"File size of {file_path_chunks}: {file_size_mb_chunks:.2f} MB)")

File size of lesson_llm_course_chunks.db: 1.94 MB)


In [ ]:
# Initialize RAGBase with the chunked index, Groq client, and specified model.
rag_helper = RAGBase(
    index=chunk_index_sql,
    llm_client=client_groq,
    # model="llama-3.3-70b-versatile",
    model="openai/gpt-oss-120b"
)
# Run the RAG query with the chunked data.
result = rag_helper.rag(query)
# Print the answer from the RAG result.
print(result.answer)
# Store the input tokens for Q5.
Q5_input_tokens = result.input_tokens
# Print the input tokens used.
print("Input tokens:", Q5_input_tokens)
# Print the output tokens used.
print("Output tokens:", result.output_tokens)

The agentic loop is built around a simple **while‑True** loop that keeps sending the conversation history back to the model and only exits when the model stops asking for a tool.

**How it works**

1. **Start the loop** – `it = 1` and `while True:` begin an infinite loop.  
2. **Call the model** – `openai_client.responses.create(...)` is invoked with the current `messages` list (the full memory) and the list of available tools (e.g., `search_tool`).  
3. **Inspect the response** – The loop iterates over `response.output`.  
   * If an item’s `type` is **`function_call`**, the code:  
     * Prints the call, runs the corresponding function (`make_call(item)`),  
     * Appends the function’s output to `messages`, and  
     * Sets `has_function_calls = True`.  
   * If the item’s `type` is **`message`**, it prints the assistant’s reply (the final answer when no function calls follow).  
4. **Update the iteration counter** – `it = it + 1`.  
5. **Check the exit condition** – After proces

In [ ]:
# Calculate and print how many times fewer input tokens were used in Q5 compared to Q3.
print(f"Q5: {int(Q3_input_tokens/Q5_input_tokens)}x fewer")

Q5: 2x fewer


## Q6. Hybrid search

Both vector and text search have their strengths and weaknesses. Vector
search matches by meaning, so it finds relevant pages even when they use
words different from the query. But it can miss exact terms like names,
codes, or rare keywords. Text search is the opposite: it nails exact words
but misses paraphrases and synonyms.

We don't have to pick one or the other - we can use both and merge their
results. This approach is called "hybrid search".

Each search produces its own ranked list, so we need a way to combine them
into one. In this homework we use Reciprocal Rank Fusion (RRF). It ignores
the raw scores from each method, which live on different scales and aren't
directly comparable. Instead, it looks only at the position of each
document in each list.

Every document scores by its position (`rank`, starting at 0) in each
list, and we sum the scores across lists with a constant `k = 60`:

```text
RRF(d) = sum over lists of  1 / (k + rank(d))
```

"Sum over lists" means we go through every ranked list and, for each list
where the document appears, add its `1 / (k + rank)` contribution. A
document found by both searches collects a score from each list, while one
found by only a single search collects just one.

The constant `k` controls how much the exact rank matters. A larger `k`
flattens the gap between positions, so the difference between rank 0 and
rank 5 counts for less. A smaller `k` does the opposite: it sharpens that
gap, so being at the top of a list matters much more.

The value 60 comes from the original RRF paper and is the usual default.
You rarely need to tune it. Lower it when only the top results matter.
Raise it to reward documents that appear across many lists, even when they
never quite reach the top.

A document that ranks well in both lists ends up higher than one that's
only strong in a single list.

```python
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]
```

Now run the query `"How do I give the model access to tools?"`
with vector and text search and fuse the results with `rrf`:

```python
results = rrf([vector_results, text_results])
```

Which file is ranked first after RRF?

* `01-agentic-rag/lessons/01-intro.md`
* `01-agentic-rag/lessons/13-function-calling.md`
* `01-agentic-rag/lessons/14-agentic-loop.md`
* `01-agentic-rag/lessons/16-other-frameworks.md`
